In [1]:
from configuracoes_notebooks import set_proj_dir
set_proj_dir()

O diretorio do seu projeto é coleta_cebrap
Caminho absoluto do diretorio encontrado C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap
Caminho no path.


In [2]:
from notebooks.jupyter import utils
from utils import (
    get_data_diretorio,
    check_crs,
    save_parquet_excel
)
from utils.downloads import download_malha_geosampa

# Mancha de Inundação (25 anos)

In [3]:
data_path= get_data_diretorio()

In [4]:
gdf_mancha_inundacao2=download_malha_geosampa('mancha_inundacao_25',
                                             data_path)

helloo
C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap\data\cache\mancha_inundacao_25.zip


In [5]:
gdf_mancha_inundacao2.shape

(30000, 11)

In [6]:
gdf_mancha_inundacao=download_malha_geosampa(
    'mancha_inundacao_25',
    data_path,
    True
)

Cuidado que talvez voce precise definir uma coluna de indice. A API retorna um documento com um erro e não dá status code correto!
Carregando arquivo em cache


In [7]:
gdf_mancha_inundacao.shape

(304724, 11)

In [8]:
gdf_mancha_inundacao.sample(2)

,cd_identif,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,cd_usuario,qt_tempo_r,dt_atualiz,sg_fonte_o,geometry
164280,362769,Morro do S,21.65,738.666,738.71,0.05,None,25,2024-08-23,FCTH,"POLYGON ((322152.202 7384630.071, 322152.202 7..."
167490,365979,Perus,86.60,741.899,742.59,0.69,None,25,2024-08-23,FCTH,"POLYGON ((321780.602 7409078.918, 321780.602 7..."


In [9]:
gdf_mancha_inundacao.columns

Index(['cd_identif', 'nm_bacia_h', 'qt_area_me', 'qt_elevaca', 'qt_cota_in',
       'qt_profund', 'cd_usuario', 'qt_tempo_r', 'dt_atualiz', 'sg_fonte_o',
       'geometry'],
      dtype='object')

Acredito que o indicador esteja se referindo apenas à área(m²) de inundação no território, mas considerando as informações de elevação, profundidade, cota de inundação* e tempo de retorno, é possível fazer análises e cálculos mais aprofundados.

*Cota de Inundação: "As cotas de inundação são valores que indicam níveis de água, que se
encontram em posições discretas sobre os rios e que são obtidas por modelos hidrodinâmicos,
estacas, sensores etc." ([ROSIM, s/d, p.2](https://files.abrhidro.org.br/Eventos/Trabalhos/154/168.pdf)).

O tempo de retorno poderia ser uma informação interessante, mas TODOS constam como 25 (anos, provavelmente), então provavelmente se refere a algo que não tempo médio de retorno da ocorrência. O mesmo vale para data de atualização.

# Padronização de nomes

In [10]:
drop_cols={
    'cd_usuario', 
    'sg_fonte_o', 
    'qt_tempo_r', 
    'dt_atualiz'
}

gdf_mancha_inundacao.rename({'cd_identif':'cd_mancha_inund'}, axis=1, inplace=True)
gdf_mancha_inundacao.drop(columns=drop_cols, axis=1, inplace=True)

In [11]:
gdf_mancha_inundacao.sample(2)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry
285386,483875,Cordeiro,21.65,735.270,735.58,0.31,"POLYGON ((328844.817 7384998.959, 328844.817 7..."
160476,358965,Morro do S,21.65,745.706,745.74,0.04,"POLYGON ((320877.202 7383660.151, 320877.202 7..."


# Conferir valor da área

Vamos conferir se há alguma mancha que tenha a área cadastra igual à área da geometria:

In [12]:
(
    gdf_mancha_inundacao
    .loc[gdf_mancha_inundacao['qt_area_me']==gdf_mancha_inundacao['geometry'].area]
)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry


É possível que isso aconteça devido ao arredondamento da área:

In [13]:
(
    gdf_mancha_inundacao
    .loc[gdf_mancha_inundacao['qt_area_me']==round(
        gdf_mancha_inundacao['geometry'].area,
        1)
    ]
).shape

(59, 7)

Há 59 casos em que a geometria cadastrada e a da área coincidem em caso de arredondamento da área da geometria.

Agora vmaos ver se nos demais casos, as diferenças são grandes demais ou não:

In [14]:
gdf_mancha_inundacao['prova_real_area'] = (
    round(
        gdf_mancha_inundacao['geometry'].area,
        1
    )-(gdf_mancha_inundacao['qt_area_me'])
)
gdf_mancha_inundacao.sample(2)


,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,prova_real_area
301566,500055,Belini e Corujas,55.424,720.405,720.46,0.06,"POLYGON ((324778.228 7394525.156, 324774.228 7...",-0.024
100667,299156,Anhangabaú,21.650,735.700,736.14,0.44,"POLYGON ((332764.961 7394651.994, 332764.961 7...",0.050


In [15]:
conferir_area = gdf_mancha_inundacao.loc[gdf_mancha_inundacao['prova_real_area']!=0.00]

In [16]:
len(gdf_mancha_inundacao)-len(conferir_area)

59

In [17]:
conferir_area.sample(2)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,prova_real_area
78796,277285,Aclimação,86.6,736.414,736.87,0.46,"POLYGON ((333912.966 7392213.768, 333912.966 7...",-1.500013e-09
180883,379372,Pirajuçara,86.6,734.091,734.94,0.85,"POLYGON ((321803.405 7389104.014, 321803.405 7...",5.999993e-09


In [18]:
conferir_area.loc[conferir_area['prova_real_area'] < 0, 'prova_real_area'] *= -1

In [19]:
conferir_area

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,geometry,prova_real_area
0,198489,Lapa,86.600000,741.619,741.69,0.070,"POLYGON ((326246.629 7396112.228, 326241.629 7...",2.199997e-09
1,198490,Lapa,86.600000,741.543,741.70,0.150,"POLYGON ((326241.629 7396120.888, 326236.629 7...",5.999993e-09
2,198491,Lapa,86.600000,741.445,741.69,0.250,"POLYGON ((326256.629 7396123.775, 326251.629 7...",6.099995e-09
3,198492,Lapa,86.600000,741.346,741.70,0.360,"POLYGON ((326261.629 7396132.435, 326256.629 7...",4.900002e-09
4,198493,Lapa,86.600000,741.486,741.69,0.204,"POLYGON ((326266.629 7396129.548, 326261.629 7...",5.200008e-09
...,...,...,...,...,...,...,...,...
304719,503208,None,89155.673004,0.000,0.00,0.000,"POLYGON ((340049.979 7403663.173, 340049.979 7...",8.915497e+04
304720,503209,None,89155.673004,0.000,0.00,0.000,"POLYGON ((340149.979 7403627.874, 340148.565 7...",8.915047e+04
304721,503210,None,22388.816773,0.000,0.00,0.000,"POLYGON ((339728.561 7404647.879, 339728.562 7...",1.677307e-02
304722,503211,None,2761.667082,0.000,0.00,0.000,"POLYGON ((339878.559 7407902.522, 339848.558 7...",3.291814e-02


In [20]:
conferir_area= conferir_area.sort_values(by=['prova_real_area'], ignore_index=True)
#conferir_area.sort_values(by=['prova_real_area'], inplace=True, ignore_index=True)

In [21]:
conferir_area['prova_real_area'] = round(conferir_area['prova_real_area'], 2)

Depois do arredondamento da diferença das áreas, aumenta o número de 0.00

# Conferir CRS

In [22]:
gdf_mancha_inundacao = check_crs(gdf_mancha_inundacao)

# Salvar GDF

In [ ]:
save_parquet_excel(
    gdf_mancha_inundacao,
    'mancha_inundacao_25',
    data_path,
    data_subpath= 'assets'
)